# EDA — Task 1: Data Cleaning & Preparation

This notebook loads the five raw NYC datasets, audits their quality, resolves
ZIP code mismatches, fills in missing demographic data, and saves the cleaned
outputs to `data/task_1/` for use by the optimisation model.

**Datasets:**
- `avg_individual_income_nyc.csv` — average individual income per ZIP
- `child_care_regulated_nyc.csv` — regulated child care facilities
- `employment_rate_nyc.csv` — employment rate per ZIP
- `population_nyc.csv` — child population by age group per ZIP
- `potential_locations_nyc.csv` — candidate sites for new facilities


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np

## 2. Load Raw Data

In [ ]:
avg_income_nyc            = pd.read_csv('../data/raw_data/avg_individual_income_nyc.csv', index_col=0)
child_care_regulated_nyc  = pd.read_csv('../data/raw_data/child_care_regulated_nyc.csv',  index_col=0)
employment_rate_nyc       = pd.read_csv('../data/raw_data/employment_rate_nyc.csv',       index_col=0)
population_nyc            = pd.read_csv('../data/raw_data/population_nyc.csv',            index_col=0)
potential_locations_nyc   = pd.read_csv('../data/raw_data/potential_locations_nyc.csv',   index_col=0)

## 3. Data Preview

Quick look at each dataset to confirm structure, column names, and data types.


In [ ]:
avg_income_nyc.head()

,zipcode,average income
0,10001,102878.033603
1,10002,59604.041165
2,10003,114273.049645
3,10004,132004.310345
4,10005,121437.713311


In [ ]:
child_care_regulated_nyc.head()

,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude
0,51349,FDC,Registration,"Valerio, Gladys",Bronx,10474,Bronx 8,0,0,0,0,6,6,40.818399,-73.888816
1,73544,SACC,Registration,YMCA OF GREATER NY,New York,10017,Manhattan 2,0,0,0,60,0,60,40.753216,-73.970794
2,108407,GFDC,License,"Almond Tree Group Family Day Care, LLC",Brooklyn,11225,Brooklyn 17,0,0,0,4,12,16,NaN,NaN
3,111953,GFDC,License,"Williams, Petal",Brooklyn,11226,Brooklyn 22,0,0,0,4,12,16,NaN,NaN
4,126425,GFDC,License,"Hernandez, Lidia",Bronx,10465,Bronx 8,0,0,0,0,12,12,40.841228,-73.823572


In [ ]:
employment_rate_nyc.head()

,zipcode,employment rate
0,10001,0.595097
1,10002,0.520662
2,10003,0.497244
3,10004,0.506661
4,10005,0.665833


In [ ]:
population_nyc.head()

,zipcode,Total,-5,6-12,13-14,15-19,20-24,25-29,30-34,35-39,40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85+
0,10001,27004,744,1255,471,1035,2296,3806,3588,2524,1702,1903,1704,1225,1323,933,815,616,488,576
1,10002,76518,2142,4645,1599,2652,4528,6988,6278,5157,4962,4822,4410,6106,4548,4815,4748,2531,2793,2794
2,10003,53877,1440,1510,477,7013,6344,7100,6427,3221,2907,1988,2698,2350,2274,2793,1854,1646,779,1056
3,10004,4579,433,262,81,108,109,601,724,490,241,313,549,279,199,173,2,15,0,0
4,10005,8801,484,318,115,53,989,2604,1144,945,685,351,652,218,85,92,66,0,0,0


In [ ]:
potential_locations_nyc.head()

,zipcode,latitude,longitude
0,10001,40.741893,-74.000140
1,10001,40.752007,-74.005436
2,10001,40.750545,-73.997147
3,10001,40.744080,-74.001932
4,10001,40.748690,-73.999341


## 4. ZIP Code Consistency Checks

### 4.1 Employment Rate vs Average Income Coverage

Before merging, we verify that `employment_rate_nyc` and `avg_income_nyc` cover
the same set of ZIP codes. Any ZIP present in one but not the other will have
missing data in the final merged model.


In [ ]:
# Cross-check ZIP codes between employment_rate_nyc and avg_income_nyc.
# Both datasets should cover the same set of zip codes; any mismatch means
# some zip codes will be missing income or employment data in the merged model.
emp_zips = set(employment_rate_nyc['zipcode'])
inc_zips = set(avg_income_nyc['zipcode'])

only_in_employment = emp_zips - inc_zips
only_in_income     = inc_zips - emp_zips
in_both            = emp_zips & inc_zips

print(f"Zip codes in employment_rate_nyc : {len(emp_zips)}")
print(f"Zip codes in avg_income_nyc      : {len(inc_zips)}")
print(f"Zip codes in both                : {len(in_both)}")
print(f"Only in employment_rate_nyc      : {sorted(only_in_employment) or 'None'}")
print(f"Only in avg_income_nyc           : {sorted(only_in_income) or 'None'}")


Zip codes in employment_rate_nyc : 179
Zip codes in avg_income_nyc      : 179
Zip codes in both                : 179
Only in employment_rate_nyc      : None
Only in avg_income_nyc           : None


### 4.2 Duplicate ZIP Codes in `population_nyc`


In [ ]:
# Sanity check: population_nyc should have exactly one row per zip code.
# Duplicate zip codes would cause double-counting when we merge datasets.
print(f"Total rows: {len(population_nyc)}")
print(f"Duplicate rows (all columns): {population_nyc.duplicated().sum()}")
print(f"Duplicate zip codes: {population_nyc.duplicated(subset='zipcode').sum()}")
print()

dupes = population_nyc[population_nyc.duplicated(subset='zipcode', keep=False)]
if dupes.empty:
    print("No duplicate zip codes found.")
else:
    print(f"Rows with duplicate zip codes ({dupes['zipcode'].nunique()} zip codes affected):")
    print(dupes.sort_values('zipcode').to_string())


Total rows: 211
Duplicate rows (all columns): 0
Duplicate zip codes: 0

No duplicate zip codes found.


## 5. Geographic Coverage Map

Interactive choropleth of child population density overlaid with existing child
care facility locations. Bubble size represents the number of facilities per ZIP.


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import requests

# --- Build population dataframe ---
pop_df = population_nyc[['zipcode', 'Total']].copy()
pop_df['zipcode_str'] = pop_df['zipcode'].astype(str)

# --- Facility count + mean coordinates per zip ---
facility_counts = (
    child_care_regulated_nyc.groupby('zipcode')
    .size()
    .reset_index(name='facility_count')
)
fac_coords = (
    child_care_regulated_nyc[['zipcode', 'latitude', 'longitude']]
    .dropna()
    .groupby('zipcode')[['latitude', 'longitude']]
    .mean()
    .reset_index()
)
fac_df = facility_counts.merge(fac_coords, on='zipcode', how='left').dropna(subset=['latitude', 'longitude'])

# --- Fetch & filter GeoJSON to only our zip codes ---
geojson_url = "https://raw.githubusercontent.com/fedhere/PUI2015_EC/master/mam1612_EC/nyc-zip-code-tabulation-areas-polygons.geojson"
nyc_geojson = requests.get(geojson_url).json()

valid_zips = set(pop_df['zipcode_str'])
nyc_geojson_filtered = {
    **nyc_geojson,
    'features': [f for f in nyc_geojson['features'] if f['properties']['postalCode'] in valid_zips]
}

# --- Choropleth: population by zip ---
fig = px.choropleth_map(
    pop_df,
    geojson=nyc_geojson_filtered,
    locations='zipcode_str',
    featureidkey='properties.postalCode',
    color='Total',
    color_continuous_scale='Blues',
    hover_data={'zipcode': True, 'Total': True, 'zipcode_str': False},
    zoom=9.5,
    center={'lat': 40.73, 'lon': -73.95},
    map_style='carto-positron',
    opacity=0.6,
    title='Population & Existing Child Care Facilities by Zip Code',
    labels={'Total': 'Population'},
)

# --- Overlay: facility dots sized by count ---
fig.add_trace(go.Scattermap(
    lat=fac_df['latitude'],
    lon=fac_df['longitude'],
    mode='markers',
    marker=dict(
        size=fac_df['facility_count'],
        sizemode='area',
        sizeref=0.4,
        sizemin=5,
        color='#e74c3c',
        opacity=0.85,
    ),
    text=fac_df['zipcode'].astype(str) + '<br>' + fac_df['facility_count'].astype(str) + ' facilities',
    hoverinfo='text',
    name='Facilities',
))

fig.update_layout(margin=dict(l=0, r=0, t=30, b=0))
fig.show(renderer='browser')

Comment on the map: For 11693 and 10464 we simply dont see the polygon because of a geojson issue but it exists in the population dataset

## 6. Facility-Level EDA

### 6.1 Geographic Spread of Facilities


In [ ]:
# List all unique ZIP codes that appear in the child care facility dataset.
# This tells us the geographic spread of existing regulated facilities.
unique_zips = child_care_regulated_nyc['zipcode'].unique()

print(f"Total Unique ZIP Codes: {len(unique_zips)}")
print("---")
print(sorted(unique_zips))


Total Unique ZIP Codes: 179
---
[np.int64(10001), np.int64(10002), np.int64(10003), np.int64(10004), np.int64(10005), np.int64(10006), np.int64(10007), np.int64(10009), np.int64(10010), np.int64(10011), np.int64(10012), np.int64(10013), np.int64(10014), np.int64(10016), np.int64(10017), np.int64(10019), np.int64(10020), np.int64(10021), np.int64(10022), np.int64(10023), np.int64(10024), np.int64(10025), np.int64(10026), np.int64(10027), np.int64(10028), np.int64(10029), np.int64(10030), np.int64(10031), np.int64(10032), np.int64(10033), np.int64(10034), np.int64(10035), np.int64(10036), np.int64(10037), np.int64(10038), np.int64(10039), np.int64(10040), np.int64(10044), np.int64(10065), np.int64(10069), np.int64(10075), np.int64(10128), np.int64(10166), np.int64(10280), np.int64(10282), np.int64(10301), np.int64(10302), np.int64(10303), np.int64(10304), np.int64(10305), np.int64(10306), np.int64(10307), np.int64(10308), np.int64(10309), np.int64(10310), np.int64(10311), np.int64(10312)

### 6.2 Missing Values (NaN Audit)

Check all four tabular datasets for missing values. Any NaNs in key columns
(income, employment rate, population) would compromise the desert classification logic.


In [ ]:
datasets = {
    'population_nyc':           population_nyc,
    'avg_income_nyc':           avg_income_nyc,
    'employment_rate_nyc':      employment_rate_nyc,
    'child_care_regulated_nyc': child_care_regulated_nyc,
}

for name, df in datasets.items():
    nan_counts = df.isnull().sum()
    nan_cols   = nan_counts[nan_counts > 0]
    total_nans = nan_counts.sum()

    print(f"{'─'*50}")
    print(f"  {name}  ({len(df)} rows, {df.shape[1]} cols)")
    print(f"  Total NaNs: {total_nans}")
    if nan_cols.empty:
        print("  No NaNs found.")
    else:
        for col, n in nan_cols.items():
            print(f"    {col:<30} {n:>6} NaNs  ({100*n/len(df):.1f}%)")
print(f"{'─'*50}")

──────────────────────────────────────────────────
  population_nyc  (211 rows, 20 cols)
  Total NaNs: 0
  No NaNs found.
──────────────────────────────────────────────────
  avg_income_nyc  (179 rows, 2 cols)
  Total NaNs: 0
  No NaNs found.
──────────────────────────────────────────────────
  employment_rate_nyc  (179 rows, 2 cols)
  Total NaNs: 0
  No NaNs found.
──────────────────────────────────────────────────
  child_care_regulated_nyc  (7740 rows, 15 cols)
  Total NaNs: 353
    school_district_name               15 NaNs  (0.2%)
    latitude                          169 NaNs  (2.2%)
    longitude                         169 NaNs  (2.2%)
──────────────────────────────────────────────────


For now the Nans we have do not affect our modelling so we can keep everything

### 6.3 ZIP Code Overlap — Facilities vs Population


In [ ]:
# Compare facility ZIP codes against population ZIP codes.
# Facilities in ZIPs that don't appear in population_nyc cannot be used in the
# model — we need to remap or drop them.
facility_zips   = set(child_care_regulated_nyc['zipcode'].dropna().astype(int))
population_zips = set(population_nyc['zipcode'].dropna().astype(int))

matching_zips    = facility_zips & population_zips
missing_from_pop = facility_zips - population_zips

print(f"Unique ZIPs in Facilities : {len(facility_zips)}")
print(f"Unique ZIPs in Population : {len(population_zips)}")
print(f"{'─'*45}")
print(f"ZIPs present in both      : {len(matching_zips)}")
print(f"ZIPs in Facilities but NOT in Population: {len(missing_from_pop)}")
if missing_from_pop:
    print(f"Missing ZIP codes: {sorted(missing_from_pop)}")


Unique ZIPs in Facilities: 179
Unique ZIPs in Population: 211
─────────────────────────────────────────────
ZIPs present in both:      178
ZIPs in Facilities but NOT in Population: 1
Missing ZIP codes: [10166]


### 6.4 Zero-Population ZIP Codes

ZIP codes with a population of zero are likely commercial buildings, airports,
or postal sorting facilities — not residential areas. We identify them here
before deciding how to handle them.


In [ ]:
# Identify zero-population zip codes
zero_pop = population_nyc[population_nyc['Total'] == 0]
print(f"Zero-population zip codes ({len(zero_pop)}):")
print(sorted(zero_pop['zipcode'].tolist()))

Zero-population zip codes (28):
[10020, 10103, 10110, 10111, 10112, 10115, 10119, 10152, 10153, 10154, 10165, 10167, 10168, 10169, 10170, 10171, 10172, 10173, 10174, 10177, 10199, 10271, 10278, 10311, 11359, 11371, 11424, 11451]


Assigning the 0 populations (buildings) to a zip_code
and the facility zip_code that did not exist in the population to an area zip_code

In [ ]:
# Remap zero-population building/facility zip codes to their surrounding area zip code
building_zip_mapping = {
    # Rockefeller Center cluster → Midtown West
    10020: 10036,   # Rockefeller Center
    10103: 10036,   # 1251 Ave of the Americas
    10110: 10036,   # 1270 Ave of the Americas
    10111: 10036,   # 30 Rockefeller Plaza
    10112: 10036,   # Rockefeller Center

    # Upper West Side
    10115: 10025,   # Riverside Church / Union Theological Seminary

    # Penn Station / MSG cluster → Chelsea/Hudson Yards
    10119: 10001,   # Penn Station
    10173: 10001,   # 1 Penn Plaza
    10199: 10001,   # Madison Square Garden / Penn Station

    # Grand Central / Park Ave cluster → Murray Hill
    10154: 10017,   # 200 Park Ave (MetLife Bldg)
    10165: 10017,   # 335 Madison Ave
    10166: 10017,   # MetLife Building (caught from facilities, not population)
    10167: 10017,   # Grand Hyatt
    10168: 10017,   # 230 Park Ave
    10169: 10017,   # 420 Lexington Ave
    10174: 10017,   # Grand Central Terminal
    10177: 10017,   # 100 Park Ave

    # Midtown East / 3rd Ave cluster → Midtown East
    10152: 10022,   # 885 Third Ave
    10153: 10022,   # 645 Fifth Ave (Olympic Tower)
    10170: 10022,   # 600 Lexington Ave
    10171: 10022,   # 757 Third Ave
    10172: 10022,   # Citigroup Center

    # Financial District
    10271: 10005,   # 1 Chase Manhattan Plaza
    10278: 10007,   # US Courthouse / Federal Building

    # Staten Island
    10311: 10314,   # Travis (small area, absorbed into 10314)

    # Queens
    11359: 11360,   # Bayside postal facility → Bayside
    11371: 11369,   # LaGuardia Airport → East Elmhurst
    11424: 11432,   # Jamaica postal facility → Jamaica
    11451: 11432,   # Jamaica government offices → Jamaica
}

# Apply to both datasets
population_nyc['zipcode']            = population_nyc['zipcode'].replace(building_zip_mapping)
child_care_regulated_nyc['zipcode']  = child_care_regulated_nyc['zipcode'].replace(building_zip_mapping)

#should be 28+1
print(f"Remapped {len(building_zip_mapping)} zip codes in population_nyc and child_care_regulated_nyc")

Remapped 29 zip codes in population_nyc and child_care_regulated_nyc


In [ ]:
# After remapping several building/PO-Box ZIPs to their residential equivalents,
# some rows in population_nyc may now share the same zip code.
# We aggregate them by summing all age-column counts to avoid double-counting.
age_cols = [c for c in population_nyc.columns if c != 'zipcode']
population_nyc = population_nyc.groupby('zipcode', as_index=False)[age_cols].sum()

print(f"Unique zip codes after aggregation: {len(population_nyc)}")
population_nyc.head()


Unique zip codes after aggregation: 183


,zipcode,Total,-5,6-12,13-14,15-19,20-24,25-29,30-34,35-39,40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85+
0,10001,27004,744,1255,471,1035,2296,3806,3588,2524,1702,1903,1704,1225,1323,933,815,616,488,576
1,10002,76518,2142,4645,1599,2652,4528,6988,6278,5157,4962,4822,4410,6106,4548,4815,4748,2531,2793,2794
2,10003,53877,1440,1510,477,7013,6344,7100,6427,3221,2907,1988,2698,2350,2274,2793,1854,1646,779,1056
3,10004,4579,433,262,81,108,109,601,724,490,241,313,549,279,199,173,2,15,0,0
4,10005,8801,484,318,115,53,989,2604,1144,945,685,351,652,218,85,92,66,0,0,0


Now lets check whether all the zip_codes in population have a corresponding avg_income and avg_employment

In [ ]:
# Check zip code overlap: population_nyc vs avg_income_nyc
pop_zips = set(population_nyc['zipcode'])
inc_zips = set(avg_income_nyc['zipcode'])

only_in_pop    = pop_zips - inc_zips
only_in_income = inc_zips - pop_zips

print(f"Unique zip codes in population_nyc  : {len(pop_zips)}")
print(f"Unique zip codes in avg_income_nyc  : {len(inc_zips)}")
print(f"Zip codes in both                   : {len(pop_zips & inc_zips)}")
print()
print(f"Only in population_nyc  : {sorted(only_in_pop) or 'None'}")
print(f"Only in avg_income_nyc  : {sorted(only_in_income) or 'None'}")

Unique zip codes in population_nyc  : 183
Unique zip codes in avg_income_nyc  : 179
Zip codes in both                   : 179

Only in population_nyc  : [10279, 11249, 11430, 11439]
Only in avg_income_nyc  : None


So 4 zip_codes in population have missing rates

### 6.5 Deep Dive — 4 ZIP Codes Missing Income & Employment Data


In [ ]:
import pgeocode
nomi = pgeocode.Nominatim('us')

deep_dive_zips = [10279, 11249, 11430, 11439]

# --- Geo lookup ---
geo = nomi.query_postal_code([str(z) for z in deep_dive_zips])
geo_info = pd.DataFrame({
    'zipcode':    deep_dive_zips,
    'place_name': geo['place_name'].values,
    'county':     geo['county_name'].values,
    'lat':        geo['latitude'].values,
    'lon':        geo['longitude'].values,
})

# --- Dataset presence ---
src = {
    'income':     set(avg_income_nyc['zipcode']),
    'employment': set(employment_rate_nyc['zipcode']),
    'facilities': set(child_care_regulated_nyc['zipcode'].dropna().astype(int)),
}
geo_info['has_income']     = geo_info['zipcode'].apply(lambda z: z in src['income'])
geo_info['has_employment'] = geo_info['zipcode'].apply(lambda z: z in src['employment'])
geo_info['has_facilities'] = geo_info['zipcode'].apply(lambda z: z in src['facilities'])

# --- Population breakdown ---
age_cols = [c for c in population_nyc.columns if c != 'zipcode']
pop_subset = population_nyc[population_nyc['zipcode'].isin(deep_dive_zips)].set_index('zipcode')

print("=" * 60)
print("DEEP DIVE: 4 ZIP CODES — only in population, missing income & employment")
print("=" * 60)

for _, row in geo_info.iterrows():
    z = row['zipcode']
    pop_row = pop_subset.loc[z] if z in pop_subset.index else None
    print(f"\n{'─'*60}")
    print(f"  ZIP: {z}  |  {row['place_name']}, {row['county']}")
    print(f"  Coords : ({row['lat']:.4f}, {row['lon']:.4f})")
    print(f"  Income data    : {'✓' if row['has_income']     else '✗ MISSING'}")
    print(f"  Employment data: {'✓' if row['has_employment'] else '✗ MISSING'}")
    print(f"  Has facilities : {'✓' if row['has_facilities'] else '✗ none'}")
    if pop_row is not None:
        print(f"  Total pop      : {int(pop_row['Total']):,}")
        print(f"  Age breakdown  :")
        for col in age_cols:
            if col != 'Total':
                print(f"    {col:>6} : {int(pop_row[col]):,}")

print(f"\n{'─'*60}")

DEEP DIVE: 4 ZIP CODES — only in population, missing income & employment

────────────────────────────────────────────────────────────
  ZIP: 10279  |  New York, New York
  Coords : (40.7127, -74.0078)
  Income data    : ✗ MISSING
  Employment data: ✗ MISSING
  Has facilities : ✗ none
  Total pop      : 161
  Age breakdown  :
        -5 : 0
      6-12 : 0
     13-14 : 0
     15-19 : 0
     20-24 : 0
     25-29 : 0
     30-34 : 51
     35-39 : 0
     40-44 : 0
     45-49 : 78
     50-54 : 0
     55-59 : 0
     60-64 : 32
     65-69 : 0
     70-74 : 0
     75-79 : 0
     80-84 : 0
       85+ : 0

────────────────────────────────────────────────────────────
  ZIP: 11249  |  Brooklyn, Kings
  Coords : (40.6451, -73.9450)
  Income data    : ✗ MISSING
  Employment data: ✗ MISSING
  Has facilities : ✓
  Total pop      : 45,087
  Age breakdown  :
        -5 : 4,774
      6-12 : 4,836
     13-14 : 1,550
     15-19 : 2,867
     20-24 : 2,429
     25-29 : 5,929
     30-34 : 5,247
     35-39 : 4,8

From our analysis we get:

| ZIP   | Area                           | Total Pop | Has Kids | Action                                                                        |
|-------|--------------------------------|-----------|----------|-------------------------------------------------------------------------------|
| 10279 | Manhattan – Financial District | 161       | No       | Drop — no kids, no income/employment data                                     |
| 11249 | Brooklyn – Williamsburg        | 45,087    | Yes      | Demographic peer proxy — income $85k, employment 65% (gentrified area)        |
| 11430 | Queens – JFK Airport           | 332       | Yes      | Transfer 37 children to nearest residential zip — airport/industrial zone, no facility viable |
| 11439 | Queens – Jamaica               | 2,140     | No       | Drop — no kids, no income/employment data                                     |

In [ ]:
# Drop zip codes with no kids or no usable income/employment data
# 11430 (JFK Airport) is handled separately — population transferred to nearest residential zip
drop_zips = [10279, 11439]
population_nyc = population_nyc[~population_nyc['zipcode'].isin(drop_zips)].reset_index(drop=True)

print(f"Dropped : {drop_zips}")
print(f"Unique zip codes in population_nyc: {len(population_nyc)}")
print(f"Remaining to resolve: [11249] → domain knowledge |  [11430] → transfer demand to nearest residential zip")

Dropped : [10279, 11439]
Unique zip codes in population_nyc: 181
Remaining to resolve: [11249] → peer proxy  |  [11430] → transfer demand to nearest residential zip


In [ ]:
# Manual income & employment assignment based on domain knowledge
# ─────────────────────────────────────────────────────────────────
# 11249 — Williamsburg (gentrified, high-income)
#   Neighboring zips are not representative; assigning peer-proxy values.
#   → income > $60k, employment ≥ 60%  ⟹  Normal-Demand (slot ratio 1/3)

manual_income      = pd.DataFrame([{'zipcode': 11249, 'average income': 85_000.0}])
manual_employment  = pd.DataFrame([{'zipcode': 11249, 'employment rate': 0.65}])

avg_income_nyc      = pd.concat([avg_income_nyc,      manual_income],     ignore_index=True)
employment_rate_nyc = pd.concat([employment_rate_nyc, manual_employment], ignore_index=True)

print("Manually assigned for 11249 (Williamsburg):")
print(f"  Average income    : $85,000")
print(f"  Employment rate   : 0.65")
print(f"\navg_income_nyc rows     : {len(avg_income_nyc)}")
print(f"employment_rate_nyc rows: {len(employment_rate_nyc)}")

Manually assigned for 11249 (Williamsburg):
  Average income    : $85,000
  Employment rate   : 0.65

avg_income_nyc rows     : 180
employment_rate_nyc rows: 180


In [ ]:
# 11430 (JFK Airport) — not a residential area.
# Its 37 children are reassigned to 11434 (Jamaica), the nearest residential zip.

nearest_zip = 11434
age_cols = [c for c in population_nyc.columns if c != 'zipcode']
jfk_row = population_nyc[population_nyc['zipcode'] == 11430][age_cols].values

if len(jfk_row) > 0:
    population_nyc.loc[population_nyc['zipcode'] == nearest_zip, age_cols] += jfk_row[0]
    population_nyc = population_nyc[population_nyc['zipcode'] != 11430].reset_index(drop=True)
    print(f"Transferred 11430 population → {nearest_zip}")
else:
    print("11430 not found in population_nyc (already removed)")

print(f"Unique zip codes in population_nyc: {len(population_nyc)}")

Transferred 11430 population → 11434
Unique zip codes in population_nyc: 180


## 7. Final ZIP Code Summary

After all remapping, aggregation, and manual assignments, we confirm that every
ZIP in `population_nyc` has a corresponding entry in `avg_income_nyc` and
`employment_rate_nyc`.


In [ ]:
# Final zip code counts across all datasets
datasets = {
    'population_nyc':           population_nyc,
    'avg_income_nyc':           avg_income_nyc,
    'employment_rate_nyc':      employment_rate_nyc,
    'child_care_regulated_nyc': child_care_regulated_nyc,
    'potential_locations_nyc':  potential_locations_nyc,
}

print("Dataset                    | Unique ZIP codes")
print("─" * 45)
for name, df in datasets.items():
    n = df['zipcode'].nunique()
    print(f"{name:<30} | {n}")

---
## End of EDA & Data Manipulation

Our final zip code list is created and all datasets are fully aligned — every zip in `population_nyc` has a corresponding entry in `avg_income_nyc`, `employment_rate_nyc`, and `child_care_regulated_nyc`.

## 8. Save Cleaned Datasets

Export all cleaned and aligned datasets to `data/task_1/` for use by the
Task 1 optimisation model and the Task 2 EDA.


In [ ]:
import os

os.makedirs('../data/task_1', exist_ok=True)

task1_saves = {
    'population_nyc.csv':           population_nyc,
    'avg_income_nyc.csv':           avg_income_nyc,
    'employment_rate_nyc.csv':      employment_rate_nyc,
    'child_care_regulated_nyc.csv': child_care_regulated_nyc,
}

for filename, df in task1_saves.items():
    df.to_csv(os.path.join('../data/task_1', filename), index=False)
    print(f"Saved {filename}  ({len(df)} rows, {df['zipcode'].nunique()} unique zips)")